#### Streamlit 기본 구조

In [1]:
import joblib

model = joblib.load("final_model.pkl")

In [3]:
import streamlit as st
import pandas as pd
import folium
from streamlit_folium import st_folium
import plotly.express as px
import joblib

2025-12-03 17:30:05.834 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


#### 페이지 선택 만들기 

In [4]:
st.set_page_config(layout="wide")

menu = st.tabs(["📍 지도 기반 분석", "📊 업종 비교 분석", "📁 상세 데이터"])

tab1, tab2, tab3 = menu

2025-12-03 17:30:08.355 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-03 17:30:08.356 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-03 17:30:08.357 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-03 17:30:08.358 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-03 17:30:08.358 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


### 지도 기반 분석 

In [7]:
streamlit run app.py

SyntaxError: invalid syntax (3737097518.py, line 1)

In [6]:
import streamlit as st
import pandas as pd
import folium
from streamlit_folium import st_folium
import plotly.express as px
import joblib

# ------------------------------------------------
# 1) 모델 & 데이터 반드시 여기서 로드해야 한다
# ------------------------------------------------

df = pd.read_csv("전처리된_최종데이터.csv")
model = joblib.load("final_logistic_model.pkl")

# feature list도 준비해야 함
feature_cols = [c for c in df.columns if c not in ["폐업여부"]]
X = df[feature_cols]

# ------------------------------------------------
# 2) Streamlit UI 시작
# ------------------------------------------------

st.set_page_config(layout="wide")
tab1, tab2, tab3 = st.tabs(["지도", "업종 비교", "데이터"])

# ------------------------------------------------
# 3) 지도 페이지(tab1)
# ------------------------------------------------

with tab1:
    st.header("1. 지도 기반 상권 및 유동 인구 분석")

    df_map = df.copy()
    df_map["pred_prob"] = model.predict_proba(X)[:, 1]

    # 지도 중심 위치
    m = folium.Map(location=[37.88, 127.73], zoom_start=12)

    for _, row in df_map.iterrows():
        if ("위도" in df_map.columns) and ("경도" in df_map.columns):
            folium.CircleMarker(
                location=[row["위도"], row["경도"]],
                radius=5,
                color="red" if row["pred_prob"] > 0.5 else "blue",
                fill=True
            ).add_to(m)

    st_folium(m, width=1000)

FileNotFoundError: [Errno 2] No such file or directory: '전처리된_최종데이터.csv'

In [5]:
with tab1:
    st.header("1. 지도 기반 상권 및 유동 인구 분석")

    df_map = df.copy()
    df_map["pred_prob"] = model.predict_proba(X)[:, 1]

    m = folium.Map(location=[37.88, 127.73], zoom_start=12)

    for _, row in df_map.iterrows():
        folium.CircleMarker(
            location=[row["위도"], row["경도"]],
            radius=5,
            color="red" if row["pred_prob"] > 0.5 else "blue",
            fill=True
        ).add_to(m)

    st_folium(m, width=1000)

2025-12-03 17:30:10.694 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-03 17:30:10.730 
  command:

    streamlit run /opt/anaconda3/lib/python3.13/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-12-03 17:30:10.731 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NameError: name 'df' is not defined

#### 업종별 비교 분석 (Plotly)

In [ ]:
with tab2:
    st.header("2. 지역별 시장 포화도 및 업종 비교")

    업종별 = df.groupby("위생업태명").agg(
        count=("위생업태명", "count"),
        폐업예측평균=("pred_prob", "mean")
    ).reset_index()

    fig = px.bar(
        업종별,
        x="위생업태명",
        y="count",
        color="폐업예측평균",
        color_continuous_scale="RdBu",
        title="업종별 업체 수 및 폐업 위험도"
    )
    st.plotly_chart(fig, use_container_width=True)

### 상세페이지 탭 

In [ ]:
with tab3:
    st.header("3. 상세 데이터 조회")
    st.dataframe(df)